In [1]:
# If required, uncomment:
# %pip install pandas numpy scikit-learn matplotlib seaborn joblib

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

RANDOM_STATE = 42
TARGET = "Condition_Class"
CLASS_NAMES = ["Excellent", "Good", "Fair", "Poor"]

In [2]:
# Shared grouped splits + the agreed 30 features (see codd_common.py).
# Never regenerate the split here: all models must use the same files.
from codd_common import load_grouped_splits, assert_feature_set

data = load_grouped_splits()
train_df, val_df, test_df = data.train_df, data.val_df, data.test_df
X_train, X_val, X_test = data.X_train, data.X_val, data.X_test
y_train, y_val, y_test = data.y_train, data.y_val, data.y_test
feature_columns = data.feature_columns
numeric_features = data.numeric_features
categorical_features = data.categorical_features

assert_feature_set(X_train.columns)
assert_feature_set(X_val.columns)
assert_feature_set(X_test.columns)
print("Features used:", feature_columns)


Grouped splits: train (11465, 30) val (2410, 30) test (2644, 30)
Image_ID overlap (must all be 0): {'train-val': 0, 'train-test': 0, 'val-test': 0}
Features: 30 (24 numeric, 6 categorical)
Features used: ['Material_Type', 'Material_Subtype', 'Source', 'Exposure_Type', 'Crack_Level', 'Surface_Damage', 'Breakage_Level', 'Discoloration_Level', 'Deformation_Level', 'Missing_Portion', 'pixel_obj_width', 'pixel_obj_height', 'relative_area', 'edge_density', 'fill_ratio', 'color_std', 'mean_brightness', 'std_brightness', 'contrast', 'texture_laplacian_var', 'dark_pixel_ratio', 'bright_pixel_ratio', 'mean_hue', 'mean_saturation', 'saturation_std', 'contour_area', 'contour_perimeter', 'circularity', 'solidity', 'extent']


In [3]:
print('Numeric:', len(numeric_features), 'Categorical:', len(categorical_features))


Numeric: 24 Categorical: 6


In [4]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [5]:
C_OPTIONS = [0.1, 1, 10, 100]
GAMMA_OPTIONS = ["scale", "auto"]

validation_results = []
candidate_models = {}

for C in C_OPTIONS:
    for gamma in GAMMA_OPTIONS:
        svm_pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", SVC(
                kernel="rbf",
                C=C,
                gamma=gamma,
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE
            ))
        ])

        svm_pipeline.fit(X_train, y_train)
        val_pred = svm_pipeline.predict(X_val)

        macro_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
        accuracy = accuracy_score(y_val, val_pred)

        key = (C, gamma)
        candidate_models[key] = svm_pipeline
        validation_results.append({
            "C": C,
            "gamma": gamma,
            "validation_accuracy": accuracy,
            "validation_macro_f1": macro_f1
        })

results_df = pd.DataFrame(validation_results).sort_values(
    "validation_macro_f1", ascending=False
)
display(results_df)

best_row = results_df.iloc[0]
best_C = float(best_row["C"])
best_gamma = best_row["gamma"]
print("Selected C:", best_C)
print("Selected gamma:", best_gamma)

C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,C,gamma,validation_accuracy,validation_macro_f1
7,100.0,auto,0.999585,0.999702
5,10.0,auto,0.999585,0.999702
6,100.0,scale,0.997925,0.985831
4,10.0,scale,0.997095,0.984900
3,1.0,auto,0.989212,0.977195
2,1.0,scale,0.987552,0.975137
0,0.1,scale,0.851867,0.747253
1,0.1,auto,0.844398,0.734709


Selected C: 100.0
Selected gamma: auto


In [6]:
X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

svm_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        kernel="rbf",
        C=best_C,
        gamma=best_gamma,
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE
    ))
])

svm_model.fit(X_train_val, y_train_val)
print("Final SVM model trained.")

C:\Users\Ryan John\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Final SVM model trained.


In [7]:
y_test_pred = svm_model.predict(X_test)

print("SVM Test Results")
print("----------------")
print("Accuracy:", round(accuracy_score(y_test, y_test_pred), 4))
print("Macro Precision:", round(precision_score(y_test, y_test_pred, average="macro", zero_division=0), 4))
print("Macro Recall:", round(recall_score(y_test, y_test_pred, average="macro", zero_division=0), 4))
print("Macro F1-score:", round(f1_score(y_test, y_test_pred, average="macro", zero_division=0), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, labels=CLASS_NAMES, zero_division=0))

SVM Test Results
----------------
Accuracy: 0.9992
Macro Precision: 0.9914
Macro Recall: 0.9994
Macro F1-score: 0.9953

Classification Report:
              precision    recall  f1-score   support

   Excellent       0.97      1.00      0.98        32
        Good       1.00      1.00      1.00       586
        Fair       1.00      1.00      1.00      1788
        Poor       1.00      1.00      1.00       238

    accuracy                           1.00      2644
   macro avg       0.99      1.00      1.00      2644
weighted avg       1.00      1.00      1.00      2644



In [8]:
cm = confusion_matrix(y_test, y_test_pred, labels=CLASS_NAMES)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("SVM – Test Confusion Matrix")
plt.tight_layout()
plt.show()

C:\Users\Ryan John\AppData\Local\Temp\ipykernel_41360\972196520.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
os.makedirs("saved_models", exist_ok=True)

MODEL_PATH = os.path.join("saved_models", "codd_svm_pipeline.joblib")
joblib.dump(svm_model, MODEL_PATH)

print("Saved SVM pipeline to:", MODEL_PATH)
print("The saved pipeline includes preprocessing and the trained SVM.")

Saved SVM pipeline to: saved_models\codd_svm_pipeline.joblib
The saved pipeline includes preprocessing and the trained SVM.
